# Train v8_fresh — High-Rank LoRA Experiments

Based on nemotron-training-notebook (v7.7). Changes:
- Data: v8_fresh JSONL zip (9 category files)
- Rank/Alpha: configurable for 4 experiments
- Everything else: identical to your proven notebook

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import subprocess, sys, os, glob
from pathlib import Path

TARGET_DIR  = "/kaggle/working/packages"
OFFLINE_DIR = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
os.makedirs(TARGET_DIR, exist_ok=True)
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)


def _pip_install(pkgs, *, no_deps=True, no_index=False, find_links=None,
                 path_arg=None, label=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", TARGET_DIR]
    if no_deps:
        cmd.append("--no-deps")
    if no_index:
        cmd.append("--no-index")
    if find_links:
        cmd += ["--find-links", find_links]
    if path_arg:
        cmd.append(path_arg)
    else:
        cmd += pkgs
    try:
        subprocess.check_call(cmd)
        if label:
            print(f"[ok] {label}")
        return True
    except Exception as e:
        if label:
            print(f"[warn] {label} failed: {e}")
        return False


def _find_wheel(pattern, search_paths):
    for base in search_paths:
        if os.path.isdir(base):
            for f in glob.glob(f"{base}/**/{pattern}", recursive=True):
                return f
    return None


# ---------- nvidia-cutlass ----------
CUTLASS_PATHS = ["/kaggle/input/datasets/rubyducklove/nvidia-cutlass"]
cutlass_wheel = _find_wheel("nvidia_cutlass-*.whl", CUTLASS_PATHS) \
             or _find_wheel("cutlass-*.whl", CUTLASS_PATHS)
CUTLASS_AVAILABLE = bool(cutlass_wheel) and _pip_install(
    [], path_arg=cutlass_wheel, label=f"nvidia-cutlass <- {cutlass_wheel}"
)

# ---------- core deps ----------
PKG_LIST = ["trl", "peft", "datasets", "bitsandbytes", "wandb", "cut-cross-entropy"]
if os.path.isdir(OFFLINE_DIR):
    _pip_install(PKG_LIST, no_index=True, find_links=OFFLINE_DIR,
                 label=f"core deps (offline): {PKG_LIST}")
else:
    _pip_install(PKG_LIST, label=f"core deps (online): {PKG_LIST}")

# ---------- Blackwell Mamba CUDA wheels ----------
WHEEL_PATHS = ["/kaggle/input/datasets/mayukh18/nemotron-packages"]
ccv_wheel  = _find_wheel("causal_conv1d-*.whl", WHEEL_PATHS)
mssm_wheel = _find_wheel("mamba_ssm-*.whl", WHEEL_PATHS)

CAUSAL_CONV1D_AVAILABLE = bool(ccv_wheel) and _pip_install(
    [], path_arg=ccv_wheel, label=f"causal_conv1d <- {ccv_wheel}"
)
MAMBA_AVAILABLE = bool(mssm_wheel) and _pip_install(
    [], path_arg=mssm_wheel, label=f"mamba_ssm <- {mssm_wheel}"
)
FAST_PATH_AVAILABLE = MAMBA_AVAILABLE and CAUSAL_CONV1D_AVAILABLE

def _resolve_pth(d):
    for pth in Path(d).glob("*.pth"):
        with pth.open() as fp:
            rel = fp.read().strip()
            p = pth.parent / rel
            if p.exists():
                sys.path.append(str(p))

_resolve_pth(TARGET_DIR)

import transformers
TRANSFORMERS_VERSION = tuple(int(x) for x in transformers.__version__.split(".")[:2])
NEW_ENOUGH = TRANSFORMERS_VERSION >= (4, 45)

os.environ["WANDB_MODE"] = "offline"

BNB_AVAILABLE = True
WANDB_AVAILABLE = True
CCE_AVAILABLE = True

for pkg, var in [("bitsandbytes", "BNB_AVAILABLE"),
                 ("wandb",        "WANDB_AVAILABLE"),
                 ("cut_cross_entropy", "CCE_AVAILABLE")]:
    try:
        __import__(pkg)
    except Exception:
        globals()[var] = False

print("=" * 60)
print(f"  Dependency status")
print("=" * 60)
print(f"  transformers     : {transformers.__version__}")
print(f"  nvidia-cutlass   : {'YES' if CUTLASS_AVAILABLE else 'NO'}")
print(f"  causal_conv1d    : {'YES' if CAUSAL_CONV1D_AVAILABLE else 'NO'}")
print(f"  mamba_ssm        : {'YES' if MAMBA_AVAILABLE else 'NO'}")
print(f"  Mamba fast path  : {'ENABLED' if FAST_PATH_AVAILABLE else 'DISABLED'}")
print(f"  cut-cross-entropy: {'YES' if CCE_AVAILABLE else 'NO'}")
print(f"  bitsandbytes     : {'YES' if BNB_AVAILABLE else 'NO'}")
print(f"  wandb            : {'YES (offline)' if WANDB_AVAILABLE else 'NO'}")

assert NEW_ENOUGH, f"transformers {transformers.__version__} too old, need >=4.45"

# Purge Kaggle utility-script mamba_ssm
_BAD_PATH_FRAGS = ("nvidia_utility_script", "nvidia-utility-script")
sys.path[:] = [p for p in sys.path if not any(b in p for b in _BAD_PATH_FRAGS)]
for _m in list(sys.modules):
    _mfile = getattr(sys.modules[_m], "__file__", "") or ""
    if any(b in _mfile for b in _BAD_PATH_FRAGS):
        del sys.modules[_m]
if TARGET_DIR in sys.path:
    sys.path.remove(TARGET_DIR)
sys.path.insert(0, TARGET_DIR)
print("[ok] purged kaggle utility-script paths")

In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import stat, shutil, zipfile, time, json, re, glob, uuid
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainerCallback, BitsAndBytesConfig,
)
from peft import (
    LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch       : {torch.__version__}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"transformers  : {transformers.__version__}")
print(f"W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}")

In [ ]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE
# ============================================================
RUN_HASH = str(uuid.uuid4())[:8]
DATASET_VERSION = 'v8-fresh'
NOTEBOOK_VERSION = 'high_rank_lora_experiment'
print(f"Critical RUN HASH: {RUN_HASH}")

# ──── CHANGE THESE PER RUN ────
LORA_RANK  = 64    # Run 1&2: 64,  Run 3&4: 128
LORA_ALPHA = 64    # Run 1: 64, Run 2: 128, Run 3: 128, Run 4: 256

WANDB_PROJECT  = "nemotron-fine-tuning"
WANDB_RUN_NAME = f"{NOTEBOOK_VERSION}-r{LORA_RANK}_a{LORA_ALPHA}-{RUN_HASH}"
WANDB_DIR      = "/kaggle/working"

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=WANDB_DIR,
        config={
            'run_hash': RUN_HASH,
            "notebook_version": NOTEBOOK_VERSION,
            "dataset_version": DATASET_VERSION,
            "model": "Nemotron-3-Nano-30B-A3B",
            "data": "all_categorical_splits_v8_fresh — real methods",
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "learning_rate": 4e-4,
            "num_epochs": 1,
            "batch_size": 1,
            "grad_accum": 4,
            "effective_batch": 4,
            "max_seq_len": 8192,
            "warmup_steps": 275,
            "scheduler": "cosine",
            "quantization": "bf16",
        },
        tags=["nemotron", RUN_HASH, DATASET_VERSION, f"r{LORA_RANK}_a{LORA_ALPHA}"],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
else:
    print("W&B not available — training metrics logged to stdout only.")

In [ ]:
# ============================================================
# 3. TRITON FIXES — rmsnorm patch + ptxas-blackwell
# ============================================================

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

if not globals().get('FAST_PATH_AVAILABLE', False):
    patched = []
    for name, mod in list(sys.modules.items()):
        nlow = name.lower()
        if not any(k in nlow for k in ("mamba", "ssm", "selective_scan", "rmsnorm")):
            continue
        if nlow.startswith("transformers"):
            continue
        try:
            if hasattr(mod, "rmsnorm_fn"):
                mod.rmsnorm_fn = _pure_rmsnorm_fn
                patched.append(name)
        except Exception:
            pass
    print(f"[ok] rmsnorm_fn patched in: {patched}" if patched else "[info] no rmsnorm to patch")
else:
    print("[info] rmsnorm patch skipped — Mamba CUDA fast path available")

# ptxas-blackwell shim
candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) \
   or (candidates[0] if candidates else None)

if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH"):
        os.environ[v] = dst
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
    except Exception as e:
        print(f"[warn] Triton cache clear: {e}")
    print(f"[ok] ptxas binary -> {dst}")
else:
    print("[warn] no ptxas binary found")

In [ ]:
# ============================================================
# 4. HYPERPARAMETERS
# ============================================================
# LORA_RANK and LORA_ALPHA set in Cell 2b (W&B cell)
LORA_DROPOUT        = 0.0
MAX_SEQ_LEN         = 8192

# ──── Auto-scale for VRAM at high rank ────
# rank 128 + bf16 base (~60GB) + 128 MoE experts = OOM on 95GB.
# Fix: quantize base to nf4 (QLoRA) → base drops to ~15GB,
# leaving plenty of room for rank 128 + MoE + full seq_len.
if LORA_RANK >= 128:
    FORCE_MODE = "nf4"
    print(f"[VRAM] rank={LORA_RANK} >= 128 → QLoRA nf4 (base ~15GB instead of ~60GB)")
else:
    FORCE_MODE = "bf16"

NUM_EPOCHS          = 1
BATCH_SIZE          = 1           # micro-batch (kept at 1 for high-rank VRAM)
GRAD_ACCUM          = 4           # effective batch = 1 * 4 = 4
LR                  = 4e-4
WARMUP_STEPS        = 275         # 5% of ~5492 steps
SAVE_EVERY_N_EPOCHS = 1
SAVE_EVERY_N_STEPS  = 1000

USE_PACKING             = False
USE_STRATIFIED_BATCHING = True
USE_CCE                 = True
USE_MAMBA_FAST_PATH     = True
MOE_LORA_MODE           = "tied"
MODE = FORCE_MODE
USE_QLORA = MODE in ("nf4", "int8")

# ──── QLoRA disables Mamba fast path ────
# Fast path kernels call F.linear() directly on raw weight tensors,
# but 4-bit weights are packed (shape 1×N). The kernel can't
# dequantize them → shape mismatch crash. Pure PyTorch path
# goes through nn.Linear.forward() which handles dequant properly.
if USE_QLORA:
    USE_MAMBA_FAST_PATH = False
    print(f"[VRAM] QLoRA mode → Mamba fast path DISABLED (incompatible with 4-bit weights)")

if USE_MAMBA_FAST_PATH and not FAST_PATH_AVAILABLE:
    print("[warn] Fast path wheels missing — forcing OFF, batch=1, accum=4")
    USE_MAMBA_FAST_PATH = False
    BATCH_SIZE = 1
    GRAD_ACCUM = 4

if USE_CCE and not CCE_AVAILABLE:
    print("[warn] CCE missing — forcing OFF")
    USE_CCE = False

MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
RUN_TAG     = f"r{LORA_RANK}_a{LORA_ALPHA}"
OUTPUT_DIR  = f"/kaggle/working/adapter_{RUN_TAG}"
CKPT_DIR    = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ──── DATASET: v8_fresh zip with 9 JSONL files ────
DATASET_ZIP = "/kaggle/input/datasets/YOUR_USERNAME/v8-fresh/all_categorical_splits_v8_fresh.zip"

CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

print("=" * 60)
print(f"  CONFIG — rank={LORA_RANK}, alpha={LORA_ALPHA}")
print("=" * 60)
print(f"  Mode             : {MODE}{'  (QLoRA nf4 — base quantized to 4-bit)' if USE_QLORA else ''}")
print(f"  MAX_SEQ_LEN      : {MAX_SEQ_LEN}")
print(f"  LR               : {LR:.1e}  (warmup {WARMUP_STEPS}, cosine)")
print(f"  Batch            : {BATCH_SIZE}x{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} eff")
print(f"  LoRA             : r={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  MoE LoRA mode    : {MOE_LORA_MODE}")
print(f"  Mamba fast path  : {USE_MAMBA_FAST_PATH}{'  (disabled for QLoRA)' if USE_QLORA else ''}")
print(f"  Cut Cross-Entropy: {USE_CCE}")
print(f"  Stratified       : {USE_STRATIFIED_BATCHING}")

In [ ]:
# ============================================================
# 5. CALLBACKS — progress + per-epoch ckpt zip + NaN auto-halt
# ============================================================
import math as _math

class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
        self.start_time = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training",
                         unit="step", dynamic_ncols=True, file=sys.stdout)
        self.start_time = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        elapsed = time.time() - self.start_time
        step    = state.global_step
        eta     = (elapsed / step) * (state.max_steps - step) if step > 0 else 0
        loss_str = (f"loss={state.log_history[-1]['loss']:.4f}"
                    if state.log_history and "loss" in state.log_history[-1] else "loss=...")
        self.pbar.set_postfix_str(f"{loss_str}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")
        self.pbar.update(1)
        sys.stdout.flush()
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()


class NaNGuardCallback(TrainerCallback):
    def __init__(self, max_consecutive=2):
        self.max_consecutive = max_consecutive
        self.bad_streak = 0
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return
        loss = logs["loss"]
        if loss is None or _math.isnan(loss) or _math.isinf(loss):
            self.bad_streak += 1
            print(f"\n[NaN GUARD] non-finite loss={loss} (streak={self.bad_streak})")
            if self.bad_streak >= self.max_consecutive:
                print(f"[NaN GUARD] HALTING training")
                control.should_training_stop = True
        else:
            self.bad_streak = 0


class CheckpointZipCallback(TrainerCallback):
    def __init__(self, ckpt_dir, output_dir, every_n=1):
        self.ckpt_dir   = ckpt_dir
        self.output_dir = output_dir
        self.every_n    = every_n
        self.epoch_losses = {}
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)
        if epoch % self.every_n != 0:
            return
        epoch_dir = os.path.join(self.output_dir, f"epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)
        cfg_path = os.path.join(epoch_dir, "adapter_config.json")
        with open(cfg_path) as f:
            cfg = json.load(f)
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)
        zip_name = f"adapter_epoch_{epoch:02d}_{RUN_HASH}.zip"
        zip_path = os.path.join(self.ckpt_dir, zip_name)
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in sorted(os.listdir(epoch_dir)):
                fp = os.path.join(epoch_dir, fname)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fname)
        recent = [h["loss"] for h in state.log_history if "loss" in h]
        avg_loss = sum(recent[-10:]) / len(recent[-10:]) if recent else float("nan")
        self.epoch_losses[epoch] = avg_loss
        zip_mb = os.path.getsize(zip_path) / 1024 / 1024
        print(f"\n[Epoch {epoch:02d}] {zip_name}  ({zip_mb:.1f} MB)  avg_loss={avg_loss:.4f}")
        if WANDB_AVAILABLE and wandb.run is not None:
            wandb.log({"epoch_checkpoint/epoch": epoch,
                       "epoch_checkpoint/avg_loss": avg_loss}, step=state.global_step)
    def print_summary(self):
        if not self.epoch_losses:
            return
        print("\n  Checkpoint loss summary:")
        best = min(self.epoch_losses, key=self.epoch_losses.get)
        for ep, loss in sorted(self.epoch_losses.items()):
            mark = " <- best" if ep == best else ""
            print(f"    epoch {ep:02d}: loss={loss:.4f}{mark}")


class TiedMoEGradCallback(TrainerCallback):
    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        try:
            _tie_grads()
        except NameError:
            pass


ckpt_callback   = CheckpointZipCallback(CKPT_DIR, OUTPUT_DIR, SAVE_EVERY_N_EPOCHS)
nan_callback    = NaNGuardCallback(max_consecutive=2)
tied_callback   = TiedMoEGradCallback()
print("Callbacks ready: LiveProgress + NaNGuard + CheckpointZip + TiedMoEGrad")

In [ ]:
# ============================================================
# 6. LOAD DATA — 9 per-category JSONL files from v8_fresh zip
# ============================================================

all_records = []

with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    for fname in CATEGORY_FILES:
        if fname not in zf.namelist():
            print(f"  [skip] {fname} not in zip")
            continue
        n = 0
        data = zf.read(fname).decode('utf-8')
        for line in data.strip().split('\n'):
            if not line.strip():
                continue
            rec = json.loads(line)
            if "category" not in rec or not rec["category"]:
                rec["category"] = fname.replace("train_cot_", "").replace(".jsonl", "")
            all_records.append(rec)
            n += 1
        print(f"  loaded {n:>5} from {fname}")

print(f"\nTOTAL records loaded: {len(all_records)}")

In [ ]:
# ============================================================
# 7. TOKENIZE & FORMAT — apply chat template, keep category labels
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

all_texts  = []
all_labels = []
fallback_template = 0

for rec in all_records:
    msgs = [m for m in rec["messages"] if m["role"] != "system"]
    try:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        fallback_template += 1
        text = (
            f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n"
            f"<|im_start|>assistant\n{msgs[-1]['content']}<|im_end|>"
        )
    all_texts.append(text)
    all_labels.append(rec["category"])

if fallback_template:
    print(f"[warn] Used fallback template for {fallback_template} records")

hf_dataset = Dataset.from_dict({"text": all_texts, "label": all_labels})
print(f"\nFormatted dataset: {len(hf_dataset)} examples\n")

from collections import Counter
dist = Counter(all_labels)
print("Category distribution:")
for name, n in dist.most_common():
    print(f"  {name:30s} {n:5d}  ({100*n/len(all_labels):.1f}%)")

print("\nSample (first 400 chars):")
print(hf_dataset[0]['text'][:400])

In [ ]:
# ============================================================
# 8. TOKEN LENGTH DIAGNOSTIC + TRUNCATE OVERSIZED
# ============================================================
print(f"Counting tokens for {len(hf_dataset)} samples...")

def get_token_length(example):
    ids = tokenizer(example['text'], truncation=False,
                    return_attention_mask=False)['input_ids']
    return {'token_len': len(ids)}

hf_dataset = hf_dataset.map(get_token_length, desc="Counting tokens")

from collections import defaultdict
import statistics

cat_lens = defaultdict(list)
for ex in hf_dataset:
    cat_lens[ex['label']].append(ex['token_len'])

print(f"\nPer-category length stats (cutoff = {MAX_SEQ_LEN}):")
print(f"  {'Category':<28} {'count':>6} {'med':>5} {'max':>5} {'>cut':>6}")
print(f"  {'-'*28} {'------':>6} {'-----':>5} {'-----':>5} {'------':>6}")

for cat in sorted(cat_lens):
    lens = sorted(cat_lens[cat])
    n = len(lens)
    med = lens[n // 2]
    mx  = lens[-1]
    over = sum(1 for l in lens if l > MAX_SEQ_LEN)
    print(f"  {cat:<28} {n:>6} {med:>5} {mx:>5} {over:>6}")

# Truncate oversized (keep tail = answer)
oversized = sum(1 for tl in hf_dataset['token_len'] if tl > MAX_SEQ_LEN)

def _tail_truncate(example):
    if example['token_len'] <= MAX_SEQ_LEN:
        return example
    ids = tokenizer(example['text'], truncation=False, return_attention_mask=False)['input_ids']
    tail_ids = ids[-MAX_SEQ_LEN:]
    example['text'] = tokenizer.decode(tail_ids, skip_special_tokens=False)
    return example

if oversized > 0:
    hf_dataset = hf_dataset.map(_tail_truncate, desc=f"Tail-truncating {oversized} oversized")
    print(f"  Tail-truncated {oversized} samples")
hf_dataset = hf_dataset.remove_columns(['token_len'])
print(f"  Final: {len(hf_dataset)} samples")

steps_estimate = len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nEstimated optimizer steps: {steps_estimate}")

In [ ]:
# ============================================================
# 9. LOAD MODEL — bf16 or QLoRA nf4 + Mamba FAST PATH
# ============================================================

flash_whl = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", flash_whl])
        print("[ok] flash_attn installed")
    except Exception as e:
        print(f"[warn] flash_attn install skipped: {e}")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
t_load = time.time()

# ──── Build quantization config for QLoRA ────
if USE_QLORA and MODE == "nf4":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,   # nested quantization saves more VRAM
    )
    print(f"Loading base model in QLoRA nf4 (double-quant) — ~15GB instead of ~60GB...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map={"":0},
        trust_remote_code=True,
        quantization_config=quant_config,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
elif USE_QLORA and MODE == "int8":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    print(f"Loading base model in int8...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map={"":0},
        trust_remote_code=True,
        quantization_config=quant_config,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
else:
    print("Loading base model in bf16...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map={"":0},
        trust_remote_code=True,
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

# Enable Mamba CUDA fast path
nemotron_mod = None
for _name, _m in list(sys.modules.items()):
    if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
        nemotron_mod = _m
        break
if nemotron_mod is None:
    for _name in list(sys.modules.keys()):
        if "nemotron_h" in _name:
            nemotron_mod = sys.modules[_name]
            break

if nemotron_mod is not None and USE_MAMBA_FAST_PATH:
    try:
        from causal_conv1d import causal_conv1d_fn
        _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
        _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
        causal_conv1d_fn(_x, _w, None, activation="silu")
        import mamba_ssm
        nemotron_mod.is_fast_path_available = True
        print(f"[OK] Mamba FAST PATH ENABLED")
    except Exception as e:
        print(f"[FAIL] Fast path kernel check failed: {e}")
        nemotron_mod.is_fast_path_available = False
elif nemotron_mod is not None:
    nemotron_mod.is_fast_path_available = False

load_min = (time.time() - t_load) / 60
vram_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nModel loaded in {load_min:.1f} min  |  VRAM: {vram_gb:.2f} / {total_gb:.1f} GB")
if USE_QLORA:
    print(f"  QLoRA mode: {MODE} — saved ~{60 - vram_gb:.0f}GB vs bf16")

In [ ]:
# ============================================================
# 10. APPLY LoRA — Attention + Mamba + (TIED) MoE
# ============================================================
from collections import Counter
import torch.nn as nn

linear_suffixes = Counter()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        suffix = name.split(".")[-1]
        linear_suffixes[suffix] += 1

print("Linear module suffixes:")
for suffix, n in sorted(linear_suffixes.items(), key=lambda x: -x[1]):
    print(f"  {suffix:30s} {n:4d}")

ATTENTION_NAMES = ["q_proj", "k_proj", "v_proj", "o_proj", "Wqkv", "qkv_proj"]
MAMBA_NAMES     = ["in_proj", "out_proj", "x_proj", "dt_proj"]
MLP_GATE_NAMES  = ["gate_proj", "gate_up_proj", "w1", "linear_fc1", "fc1"]
MLP_UP_NAMES    = ["up_proj", "w3", "wi_1"]
MLP_DOWN_NAMES  = ["down_proj", "w2", "linear_fc2", "fc2", "wo"]
EXCLUDE         = {"lm_head", "embed_tokens", "shared", "router", "score", "classifier"}

LORA_TARGET_MODULES = []
seen = set()

def add_if_present(names, label):
    added = []
    for n in names:
        if n in linear_suffixes and n not in seen and n not in EXCLUDE:
            LORA_TARGET_MODULES.append(n)
            seen.add(n)
            added.append(n)
    if added:
        print(f"  {label:18s}: {added}")
    return added

print("\nTarget module selection:")
add_if_present(ATTENTION_NAMES, "attention")
add_if_present(MAMBA_NAMES,     "mamba")
if MOE_LORA_MODE in ("tied", "full"):
    add_if_present(MLP_GATE_NAMES, "moe gate")
    add_if_present(MLP_UP_NAMES,   "moe up")
    add_if_present(MLP_DOWN_NAMES, "moe down")
    print(f"  {'MoE strategy':<18s}: {MOE_LORA_MODE.upper()}")

print(f"\nFinal LoRA targets: {LORA_TARGET_MODULES}")
assert LORA_TARGET_MODULES

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()

# fp32 LoRA + bf16 base
print("\nCasting LoRA params -> fp32...")
n_lora_fp32 = 0
for name, param in model.named_parameters():
    if ".lora_" in name:
        param.data = param.data.to(torch.float32)
        n_lora_fp32 += 1
print(f"  LoRA params cast to fp32: {n_lora_fp32}")

# MoE weight tying
moe_tied_params = []
if MOE_LORA_MODE == "tied":
    w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
    w2_proj_names = ("down_proj", ".w2.")
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if ".experts." not in name or ".lora_" not in name:
            continue
        is_w1 = any(p in name for p in w1_proj_names)
        is_w2 = any(p in name for p in w2_proj_names)
        is_A = ".lora_A." in name
        is_B = ".lora_B." in name
        should_tie = (is_w1 and is_A) or (is_w2 and is_B)
        if not should_tie:
            continue
        if param.dim() < 2 or param.shape[0] <= 1:
            continue
        moe_tied_params.append(param)

    def _tie_param_init():
        with torch.no_grad():
            for p in moe_tied_params:
                mean = p.data.mean(dim=0, keepdim=True)
                p.data.copy_(mean.expand_as(p.data))

    def _tie_grads():
        with torch.no_grad():
            for p in moe_tied_params:
                if p.grad is None:
                    continue
                grad_sum = p.grad.sum(dim=0, keepdim=True)
                p.grad.copy_(grad_sum.expand_as(p.grad))

    print(f"\n[MoE TYING] {len(moe_tied_params)} params to tie")
    _tie_param_init()
    print("  Expert slices initialized to mean (TIED)")
else:
    def _tie_grads():
        pass

model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable params: {trainable/1e6:.1f}M")
vram_after_lora = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after LoRA: {vram_after_lora:.2f} GB")

try:
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception:
    pass

In [ ]:
# ============================================================
# 11. TRAINING — Stratified SFT + CCE loss + AdamW 8-bit
# ============================================================
from torch.utils.data import Sampler
import random as _random

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
RESUME_FROM_CHECKPOINT = None

# ──── 8-bit AdamW to fit high-rank LoRA ────
# At rank 64/128 with 128 MoE experts, full-precision AdamW optimizer
# states blow VRAM. 8-bit AdamW halves optimizer memory (~3-5GB saved).
USE_ADAMW_8BIT = BNB_AVAILABLE and LORA_RANK >= 64
if USE_ADAMW_8BIT:
    import bitsandbytes as bnb
    print(f"[ok] Using AdamW 8-bit (rank={LORA_RANK} >= 64, saves ~3-5GB VRAM)")
else:
    print(f"[ok] Using AdamW fp32 (rank={LORA_RANK})")

# CCE forward patch
if USE_CCE:
    try:
        from cut_cross_entropy import linear_cross_entropy
        _base = model
        while hasattr(_base, "model"):
            _base = _base.model
        if hasattr(_base, "backbone") and hasattr(_base, "lm_head"):
            _lm_head_module = _base.lm_head
            _orig_forward   = _base.forward

            def _cce_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
                backbone_out = _base.backbone(
                    input_ids=input_ids, attention_mask=attention_mask,
                    **{k: v for k, v in kwargs.items()
                       if k in ("position_ids", "past_key_values", "use_cache",
                                "inputs_embeds", "cache_position")},
                )
                hidden_states = backbone_out[0]
                if hasattr(_lm_head_module, "base_layer"):
                    base_w = _lm_head_module.base_layer.weight
                    if hasattr(_lm_head_module, "lora_A") and "default" in _lm_head_module.lora_A:
                        lora_A = _lm_head_module.lora_A["default"].weight
                        lora_B = _lm_head_module.lora_B["default"].weight
                        scaling = _lm_head_module.scaling["default"]
                        lm_weight = base_w + scaling * (lora_B @ lora_A)
                    else:
                        lm_weight = base_w
                else:
                    lm_weight = _lm_head_module.weight
                if labels is not None:
                    shift_hidden = hidden_states[..., :-1, :].contiguous()
                    shift_labels = labels[..., 1:].contiguous()
                    valid = shift_labels != -100
                    if valid.any():
                        loss = linear_cross_entropy(
                            shift_hidden, lm_weight,
                            shift_labels.masked_fill(~valid, 0), reduction="none",
                        )
                        loss = (loss * valid.float()).sum() / valid.float().sum().clamp(min=1)
                    else:
                        loss = shift_hidden.sum() * 0.0
                else:
                    loss = None
                from transformers.modeling_outputs import CausalLMOutputWithPast
                return CausalLMOutputWithPast(
                    loss=loss, logits=None,
                    past_key_values=getattr(backbone_out, "past_key_values", None),
                )

            _base.forward = _cce_forward
            print("[OK] CCE forward patched (~17GB saved)")
        else:
            print("[warn] CCE patch skipped: model layout missing .backbone/.lm_head")
            USE_CCE = False
    except Exception as e:
        print(f"[warn] CCE failed: {e}")
        USE_CCE = False

# Stratified sampler
def build_stratified_index_order(labels, chunk_size, seed=0):
    buckets = {}
    for i, lbl in enumerate(labels):
        buckets.setdefault(lbl, []).append(i)
    rng = _random.Random(seed)
    for lbl in buckets:
        rng.shuffle(buckets[lbl])
    order = []
    active = list(buckets.keys())
    rng.shuffle(active)
    while active:
        next_active = []
        for lbl in active:
            take = buckets[lbl][:chunk_size]
            buckets[lbl] = buckets[lbl][chunk_size:]
            order.extend(take)
            if buckets[lbl]:
                next_active.append(lbl)
        active = next_active
    return order

class PrecomputedOrderSampler(Sampler):
    def __init__(self, labels, chunk_size, num_epochs, base_seed=1337):
        self.labels = list(labels)
        self.chunk_size = chunk_size
        self.num_epochs = num_epochs
        self.base_seed = base_seed
        self.epoch = 0
        self._current_order = build_stratified_index_order(self.labels, chunk_size, seed=base_seed)
    def set_epoch(self, epoch):
        self.epoch = epoch
        self._current_order = build_stratified_index_order(self.labels, self.chunk_size, seed=self.base_seed + epoch)
    def __iter__(self):
        return iter(self._current_order)
    def __len__(self):
        return len(self.labels)

# compute_loss override for CCE
def _cce_compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    outputs = model(**inputs)
    if outputs.loss is not None:
        loss = outputs.loss
    else:
        loss = torch.zeros((), device=next(model.parameters()).device, requires_grad=True)
    return (loss, outputs) if return_outputs else loss

def _create_optimizer_fn(self_trainer):
    """Create AdamW optimizer — 8-bit for high rank, fp32 for rank<=32."""
    if self_trainer.optimizer is None:
        decay_params = [p for p in self_trainer.model.parameters() if p.requires_grad]
        total_m = sum(p.numel() for p in decay_params) / 1e6
        if USE_ADAMW_8BIT:
            self_trainer.optimizer = bnb.optim.AdamW8bit(
                decay_params, lr=self_trainer.args.learning_rate,
                betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
            )
            print(f"[ok] Optimizer: AdamW 8-bit on {total_m:.1f}M params (saves ~{total_m*4/1024:.1f}GB)")
        else:
            self_trainer.optimizer = torch.optim.AdamW(
                decay_params, lr=self_trainer.args.learning_rate,
                betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
            )
            print(f"[ok] Optimizer: AdamW fp32 on {total_m:.1f}M params")
    return self_trainer.optimizer

class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_labels=None, chunk_size=16, num_epochs=3, **kwargs):
        self._strat_labels = stratified_labels
        self._strat_chunk  = chunk_size
        self._strat_epochs = num_epochs
        super().__init__(*args, **kwargs)
    def _get_train_sampler(self, *args, **kwargs):
        if self._strat_labels is None:
            return super()._get_train_sampler(*args, **kwargs)
        return PrecomputedOrderSampler(self._strat_labels, self._strat_chunk, self._strat_epochs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return _cce_compute_loss(self, model, inputs, return_outputs, num_items_in_batch)
    def create_optimizer(self):
        return _create_optimizer_fn(self)

class _PlainTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return _cce_compute_loss(self, model, inputs, return_outputs, num_items_in_batch)
    def create_optimizer(self):
        return _create_optimizer_fn(self)

# SFTConfig
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="steps",
    save_steps=SAVE_EVERY_N_STEPS,
    save_total_limit=1,
    save_only_model=True,
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=USE_PACKING,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=16,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=4,
    remove_unused_columns=False,
)

labels_for_sampler = list(hf_dataset['label']) if USE_STRATIFIED_BATCHING else None

callbacks = [LiveProgressCallback(), nan_callback, ckpt_callback]
if MOE_LORA_MODE == "tied":
    callbacks.append(tied_callback)

if USE_STRATIFIED_BATCHING:
    print(f"Using STRATIFIED batching (chunk = {EFFECTIVE_BATCH})")
    trainer = StratifiedSFTTrainer(
        model=model, train_dataset=hf_dataset,
        processing_class=tokenizer, args=training_args,
        callbacks=callbacks, stratified_labels=labels_for_sampler,
        chunk_size=EFFECTIVE_BATCH, num_epochs=NUM_EPOCHS,
    )
else:
    print("Using standard RANDOM batching")
    trainer = _PlainTrainer(
        model=model, train_dataset=hf_dataset,
        processing_class=tokenizer, args=training_args,
        callbacks=callbacks,
    )

print("=" * 60)
print(f"  Training: rank={LORA_RANK}, alpha={LORA_ALPHA}")
print("=" * 60)
print(f"  Samples       : {len(hf_dataset)}  |  epochs={NUM_EPOCHS}")
print(f"  Batch         : {BATCH_SIZE}x{GRAD_ACCUM} = {EFFECTIVE_BATCH} eff")
print(f"  LR            : {LR:.1e}  warmup={WARMUP_STEPS}  cosine")
print(f"  LoRA          : r={LORA_RANK}, alpha={LORA_ALPHA}, MoE={MOE_LORA_MODE}")
print(f"  Optimizer     : {'AdamW 8-bit' if USE_ADAMW_8BIT else 'AdamW fp32'}")
print(f"  CCE           : {'YES' if USE_CCE else 'NO'}")
print(f"  Fast path     : {'YES' if (nemotron_mod and nemotron_mod.is_fast_path_available) else 'NO'}")
print(f"  W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}\n")

t0 = time.time()
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
elapsed_hrs = (time.time() - t0) / 3600
print(f"\nTraining complete! Time: {elapsed_hrs:.2f} hrs")
ckpt_callback.print_summary()

peak_vram = torch.cuda.max_memory_allocated() / 1e9
print(f"\nPeak VRAM: {peak_vram:.2f} GB / 95 GB")

In [ ]:
# ============================================================
# 12. SAVE FINAL ADAPTER
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)

config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

adapter_config["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
adapter_config["_custom_run_hash"] = RUN_HASH
adapter_config["_dataset_version"] = DATASET_VERSION
with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"r / alpha : {adapter_config.get('r')} / {adapter_config.get('lora_alpha')}")
print(f"targets   : {adapter_config.get('target_modules')}")

try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"), framework="pt") as f:
        keys = list(f.keys())
        norms = [f.get_tensor(k).norm().item() for k in keys[:5]]
    print(f"\nAdapter tensors: {len(keys)} params")
    print(f"First 5 norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: Norms near 0 — adapter may be untrained!")
    else:
        print("Adapter looks healthy.")
except Exception as e:
    print(f"Could not verify: {e}")

print(f"\nFiles in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  {fname}  ({os.path.getsize(fpath)/1024/1024:.2f} MB)")

In [ ]:
run_metadata = {
    "run_hash": RUN_HASH,
    "notebook_version": NOTEBOOK_VERSION,
    "dataset_version": DATASET_VERSION,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LR,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}
with open(os.path.join(OUTPUT_DIR, "run_metadata.json"), "w") as f:
    json.dump(run_metadata, f, indent=2)
print(f"Metadata saved: {run_metadata}")

In [ ]:
# ============================================================
# 13. ZIP FINAL ADAPTER
# ============================================================
ZIP_PATH = f"/kaggle/working/adapter_{RUN_TAG}_{RUN_HASH}.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)
            file_count += 1

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"adapter zip: {ZIP_PATH} ({zip_mb:.1f} MB, {file_count} files)")

# List epoch checkpoints
ckpt_zips = sorted([f for f in os.listdir(CKPT_DIR) if f.endswith(".zip")])
print(f"\nPer-epoch checkpoints in {CKPT_DIR}:")
for zname in ckpt_zips:
    zpath = os.path.join(CKPT_DIR, zname)
    mb = os.path.getsize(zpath) / 1024 / 1024
    print(f"  {zname:<40} {mb:6.1f} MB")

print(f"\nExperiment: {RUN_TAG}")
print("Done! Ready to submit.")